In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Q3 - Superimpose flag on cricket turf

### Click 4 corners using the provided code
Click 4 corners: top-left, top-right, bottom-right, bottom-left.

In [ ]:
# code from Listing 1
points = []
def mouse_callback(event, x, y, flags, param):
    global points, img_display
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            print(f"Point {len(points)}: ({x}, {y})")
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", img_display)
            if len(points) == 4:
                print("\nFour points selected:")
                for i, p in enumerate(points):
                    print(f"P{i+1}: {p}")
                print("Press any key to exit.")
img = cv2.imread("turf.jpg")
if img is None:
    raise FileNotFoundError("Image not found.")
img_display = img.copy()
cv2.namedWindow("Image")
cv2.setMouseCallback("Image", mouse_callback)
cv2.imshow("Image", img_display)
cv2.waitKey(0)
cv2.destroyAllWindows()
points = np.array(points, dtype=np.float32)
print("\nFinal array of selected points:")
print(points)

In [ ]:
dst_pts = points.copy()
vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
pts_int = dst_pts.astype(int)
for pt in pts_int:
    cv2.circle(vis, tuple(pt), 8, (255, 0, 0), -1)
for i in range(4):
    cv2.line(vis, tuple(pts_int[i]), tuple(pts_int[(i+1)%4]), (255, 0, 0), 2)
plt.figure(figsize=(10, 6))
plt.imshow(vis)
plt.title('selected corners')
plt.axis('off')
plt.show()

### Load flag

In [ ]:
flag = cv2.imread('sl_flag.png')
flag_rgb = cv2.cvtColor(flag, cv2.COLOR_BGR2RGB)
fh, fw = flag.shape[:2]
print(f'flag size: {fw} x {fh}')
plt.figure(figsize=(6, 3))
plt.imshow(flag_rgb)
plt.title('flag')
plt.axis('off')
plt.show()

### Homography and warp

In [ ]:
src_pts = np.array([[0, 0], [fw-1, 0], [fw-1, fh-1], [0, fh-1]], dtype=np.float32)
H, status = cv2.findHomography(src_pts, dst_pts)
print('Homography matrix:')
print(H)

In [ ]:
warped = cv2.warpPerspective(flag, H, (img.shape[1], img.shape[0]))
mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
cv2.fillConvexPoly(mask, dst_pts.astype(int), 255)
result = img.copy()
result[mask == 255] = warped[mask == 255]
alpha = 0.6
blended = img.copy()
blended[mask == 255] = cv2.addWeighted(img, 1-alpha, warped, alpha, 0)[mask == 255]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[0].set_title('full replacement')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
axes[1].set_title('alpha blended (60%)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Comparing different alpha values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
alphas = [0.3, 0.6, 0.9]
for ax, alpha in zip(axes, alphas):
    blend = img.copy()
    blend[mask == 255] = cv2.addWeighted(img, 1-alpha, warped, alpha, 0)[mask == 255]
    ax.imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
    ax.set_title(f'alpha = {alpha}')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Lower alpha = more turf visible, higher alpha = more flag visible')
print('0.6 gives a good balance')